# 공매 → 경매 보정 (코랩)
공매 낙찰가 모델의 예측을 **경매 기준으로 바꾸고**, 매각통계와 경매 220건으로 평가해요.

**먼저 필요한 것**
- `bid_model_colab.ipynb`를 8단계(최종 모델 저장)까지 실행해서 `data/model_out/`에 모델이 있어야 해요.
- 드라이브 `신보프로젝트/` 안에 `매각통계/`(12개 파일), `data/경매_매각사례_시험셋.xlsx`, `data/train_features_20261105.csv`, `지역별 지가지수, 지가변동률/`이 있어야 해요. (이미 올려 뒀어요)

**실행**: 런타임 → 모두 실행

## 준비. 드라이브 연결 + 패키지

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # 매번 새로 연결 (새로 올린 폴더도 보이게)

!pip install -q lightgbm openpyxl
import lightgbm
print("lightgbm", lightgbm.__version__)

Mounted at /content/drive
lightgbm 4.6.0


## 0단계. 설정
폴더 경로와 극단값 기준(공매 학습과 동일한 0.05~1.5)을 정해요.

In [ ]:
import os, glob, json, re, warnings, unicodedata
import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings("ignore")

# ▼ 구글 드라이브 경로 (다르면 여기만 고치기)
def nfc(x): return unicodedata.normalize("NFC", x)

def find_project(name="신보프로젝트"):
    """내 드라이브 → 바로가기(공유받은 폴더) 순서로 '신보프로젝트' 폴더를 찾음"""
    cands = ["/content/drive/MyDrive/" + name]
    cands += glob.glob("/content/drive/MyDrive/*/" + name)
    cands += glob.glob("/content/drive/.shortcut-targets-by-id/*/" + name)
    for root in ["/content/drive/MyDrive", "/content/drive/.shortcut-targets-by-id"]:   # 한글 형식 차이 대비
        if os.path.isdir(root):
            for d in os.listdir(root):
                full = os.path.join(root, d)
                if nfc(d) == name: cands.append(full)
                elif os.path.isdir(full):
                    cands += [os.path.join(full, x) for x in os.listdir(full) if nfc(x) == name]
    for c in cands:
        if os.path.isdir(c) and any(nfc(x) == "data" for x in os.listdir(c)):
            return c
    raise FileNotFoundError("'신보프로젝트' 폴더를 못 찾았어요. 주연님 계정으로 열었는지, "
                            "공유받은 계정이면 '내 드라이브에 바로가기 추가'를 했는지 확인하세요.")

def sub(parent, name):
    """한글 형식이 달라도 하위 폴더를 찾음"""
    for x in os.listdir(parent):
        if nfc(x) == name:
            return os.path.join(parent, x)
    return os.path.join(parent, name)

PROJECT_DIR = find_project()
DATA_DIR  = sub(PROJECT_DIR, "data")                          # train_features, 경매 시험셋, model_out
STAT_DIR  = sub(PROJECT_DIR, "매각통계")                       # 법원 용도별 매각통계 12개
MODEL_DIR = sub(DATA_DIR, "model_out")                        # bid_model_colab.ipynb 8단계에서 저장한 모델
OUT_DIR   = os.path.join(DATA_DIR, "calib_out")
os.makedirs(OUT_DIR, exist_ok=True)

CAL_END = "2026-08-31"          # 보정 기간 = 공매 train 기간 (2024.11 ~ 2026.08)
QS = [0.1, 0.5, 0.9]
SHRINK_K = 50                   # 건수 적은 용도의 계수를 전체 계수 쪽으로 당기는 강도 (공매 n건이 50건이면 반반)
Y_RANGE = (0.05, 1.5)           # 낙찰가율 극단값 기준 — 공매 학습과 똑같이

USAGES = ["아파트", "기타주거", "상가·업무", "공장·창고", "대지", "농지·임야"]
print("프로젝트 폴더:", PROJECT_DIR)
print("매각통계:", len([f for f in os.listdir(STAT_DIR) if f.endswith(".csv")]) if os.path.isdir(STAT_DIR) else "폴더 없음 ⚠️", "개")
print("모델:", sorted(os.listdir(MODEL_DIR)) if os.path.isdir(MODEL_DIR) else "없음 ⚠️ (bid_model_colab 8단계 먼저)")

프로젝트 폴더: /content/drive/MyDrive/신보프로젝트
매각통계: 12 개
모델: ['lgb_q10.txt', 'lgb_q50.txt', 'lgb_q90.txt', 'model_meta.json', 'test_predictions.csv']


## 1단계. 경매 매각가율
법원 매각통계를 용도 6묶음으로 묶고 **Σ매각가 ÷ Σ감정가**로 구해요. 기간: 보정(2024.11~2026.08), 9월(val), 10월(test).

In [ ]:
# 경매 매각통계 행 → 우리 용도 6묶음
STAT_MAP = {"아파트":    [("아파트", None)],
            "기타주거":  [("단독주택,다가구주택", "소계"), ("연립주택,다세대", None)],
            "상가·업무": [("상가,오피스텔,근린시설", "소계")],
            "대지":      [("대지,임야,전답", "대지")],
            "농지·임야": [("대지,임야,전답", "임야"), ("대지,임야,전답", "전답")],
            "공장·창고": [("기타", None)]}          # 전국 파일에만 있음 (대구·경북 파일엔 '기타' 행 없음)

def _num(x):
    return pd.to_numeric(str(x).replace(",", "").replace("%", ""), errors="coerce")

def read_stat(path):
    d = pd.read_csv(path, encoding="utf-8-sig")
    d.columns = ["대", "소", "경매건수", "매각건수", "감정가", "매각가", "매각율", "매각가율"]
    d["대"] = d["대"].ffill()
    for c in ["경매건수", "매각건수", "감정가", "매각가"]:
        d[c] = d[c].map(_num)
    return d

def stat_by_usage(files):
    """여러 파일(기간)을 금액으로 합쳐 용도별 (감정가합, 매각가합, 매각건수)"""
    acc = {u: [0.0, 0.0, 0] for u in USAGES}
    found = {u: False for u in USAGES}
    for f in files:
        d = read_stat(f)
        for u, keys in STAT_MAP.items():
            for a, b in keys:
                r = d[(d["대"] == a) & ((d["소"] == b) if b else d["소"].isna())]
                if len(r):
                    acc[u][0] += r["감정가"].sum(); acc[u][1] += r["매각가"].sum(); acc[u][2] += int(r["매각건수"].sum())
                    found[u] = True
    return pd.DataFrame({u: {"경매율": (v[1] / v[0]) if v[0] > 0 else np.nan, "경매_매각건수": v[2]}
                         for u, v in acc.items() if found[u]}).T

def stat_files(period, regions):
    out = []
    for f in glob.glob(os.path.join(STAT_DIR, "*.csv")):
        name = nfc(os.path.basename(f))
        if name.startswith(period + " ") and any(f" {r} " in name for r in regions):
            out.append(f)
    return sorted(out)

AUC = {}
for label, periods in [("보정", ["24.11~25.10", "25.11~26.08"]), ("9월", ["26.9"]), ("10월", ["26.10"])]:
    for reg_label, regs in [("전국", ["전국"]), ("대구경북", ["대구", "경북"])]:
        files = sum([stat_files(p, regs) for p in periods], [])
        if not files:
            raise FileNotFoundError(f"{label} {reg_label} 매각통계 파일을 못 찾았어요. 폴더 안 파일: "
                                    f"{[nfc(x) for x in os.listdir(STAT_DIR)]}")
        AUC[(label, reg_label)] = stat_by_usage(files)
        print(f"{label} {reg_label}: 파일 {len(files)}개")

auc_tbl = pd.concat({f"{k[0]} {k[1]}": v["경매율"] for k, v in AUC.items()}, axis=1).reindex(USAGES)
print("\n경매 매각가율 (금액 가중)"); print((auc_tbl * 100).round(1).to_string())

보정 전국: 파일 2개
보정 대구경북: 파일 4개
9월 전국: 파일 1개
9월 대구경북: 파일 2개
10월 전국: 파일 1개
10월 대구경북: 파일 2개

경매 매각가율 (금액 가중)
       보정 전국  보정 대구경북  9월 전국  9월 대구경북  10월 전국  10월 대구경북
아파트     84.3     78.5   84.2     81.5    82.9      77.5
기타주거    65.4     53.3   60.0     47.5    59.6      44.3
상가·업무   54.6     49.1   49.6     46.2    47.0      42.8
공장·창고   57.3      NaN   51.6      NaN    48.2       NaN
대지      51.9     48.1   52.5     34.0    49.2      38.3
농지·임야   45.2     41.9   40.5     34.5    35.2      29.7


## 2단계. 공매 낙찰가율
같은 기간·같은 용도·같은 방식(금액 가중)으로 공매 쪽을 구해요. 재산유형별(압류 / 기타일반 / 전체)로 따로 계산해요.

In [ ]:
def usage6(m, s):
    s = str(s)
    if s == "아파트": return "아파트"
    if m == "주거용건물" or s == "주/상용건물": return "기타주거"
    if s in ("공장시설", "기타제조공장", "창고시설", "공장용지", "창고용지"): return "공장·창고"
    if m == "토지":
        return "대지" if s in ("대지", "잡종지", "주차장", "주유소용지", "종교용지", "수도용지", "기타토지",
                               "도로", "공원", "구거", "하천", "묘지", "염전") else "농지·임야"
    return "상가·업무"

path = sorted(glob.glob(os.path.join(DATA_DIR, "train_features_*.csv")))[-1]
g = pd.read_csv(path, low_memory=False)
g["opbd_date"] = pd.to_datetime(g["opbd_date"], errors="coerce")
g = g.dropna(subset=["opbd_date", "y_ratio", "apslEvlAmt"]).sort_values("opbd_date").drop_duplicates("cltrMngNo", keep="last")
g = g[g["y_ratio"].between(*Y_RANGE)].copy()
g["usage6"] = [usage6(m, s) for m, s in zip(g["cltrUsgMclsCtgrNm"], g["cltrUsgSclsCtgrNm"])]
g["prpt"] = g["prptDivNm"].map({"압류재산": "압류", "기타일반재산": "기타일반"}).fillna("기타")
g["낙찰가"] = g["y_ratio"] * g["apslEvlAmt"]

def gongmae_rate(frame):
    t = frame.groupby("usage6").agg(감정가=("apslEvlAmt", "sum"), 낙찰가=("낙찰가", "sum"), n=("y_ratio", "size"))
    t["공매율"] = t["낙찰가"] / t["감정가"]
    return t[["공매율", "n"]].reindex(USAGES)

def period_mask(frame, start, end):
    return (frame["opbd_date"] >= pd.Timestamp(start)) & (frame["opbd_date"] <= pd.Timestamp(end))

PERIODS = {"보정": ("2024-11-01", CAL_END), "9월": ("2026-09-01", "2026-09-30"), "10월": ("2026-10-01", "2026-10-31")}
OPTIONS = {"A 압류 기준": lambda f: f[f.prpt == "압류"],
           "B 기타일반 기준": lambda f: f[f.prpt == "기타일반"],
           "C 전체": lambda f: f}

GONG = {(per, opt): gongmae_rate(sel(g[period_mask(g, *PERIODS[per])])) for per in PERIODS for opt, sel in OPTIONS.items()}
print("공매 낙찰가율 (보정 기간, 금액 가중)")
print(pd.concat({opt: (GONG[("보정", opt)]["공매율"] * 100).round(1).astype(str) + "% (" +
                      GONG[("보정", opt)]["n"].fillna(0).astype(int).astype(str) + ")" for opt in OPTIONS}, axis=1).to_string())
pa = g[g.prpt == "압류"]
print(f"\n※ 보정 기간 안 압류재산은 {pa[pa.opbd_date <= CAL_END].opbd_date.min():%Y-%m-%d} ~ {CAL_END} 뿐 → 계수가 불안정할 수 있음")

공매 낙찰가율 (보정 기간, 금액 가중)
            A 압류 기준     B 기타일반 기준          C 전체
usage6                                         
아파트      81.0% (73)   77.3% (566)   77.7% (639)
기타주거    43.7% (175)   58.2% (758)   56.8% (933)
상가·업무   27.1% (120)  71.0% (1259)  70.4% (1379)
공장·창고     38.2% (9)    56.0% (52)    55.9% (61)
대지       32.7% (72)      nan% (0)    32.7% (72)
농지·임야   23.2% (259)      nan% (0)   23.2% (259)

※ 보정 기간 안 압류재산은 2026-07-23 ~ 2026-08-31 뿐 → 계수가 불안정할 수 있음


## 3단계. 보정 계수 = 경매 ÷ 공매
전국끼리 비교해요(지역 차이는 공매 모델이 이미 반영). 공매 건수가 적은 용도는 전체 계수 쪽으로 당겨요.

In [ ]:
# 지역 차이는 공매 모델이 이미 피처(시도·대구경북)로 반영 → 계수는 '공매→경매' 차이만 담도록 전국끼리 비교
auc_cal = AUC[("보정", "전국")]

def make_factors(opt):
    gr = GONG[("보정", opt)]
    both = gr.join(auc_cal, how="left")
    ok = both.dropna(subset=["공매율", "경매율"])
    # 전체 계수: 용도별 경매 금액 비중으로 가중한 경매율 ÷ 같은 가중의 공매율
    w = ok["경매_매각건수"]
    f_all = (ok["경매율"] * w).sum() / (ok["공매율"] * w).sum()
    out = {}
    for u in USAGES:
        r = both.loc[u]
        if pd.notna(r["공매율"]) and pd.notna(r["경매율"]) and r["n"] > 0:
            raw = r["경매율"] / r["공매율"]
            out[u] = {"원계수": raw, "공매n": int(r["n"]),
                      "보정계수": (r["n"] * raw + SHRINK_K * f_all) / (r["n"] + SHRINK_K)}
        else:
            out[u] = {"원계수": np.nan, "공매n": 0, "보정계수": f_all}          # 공매 자료 없는 용도는 전체 계수
    t = pd.DataFrame(out).T
    t.loc["(전체)"] = [f_all, int(gr["n"].sum()), f_all]
    return t

FACT = {opt: make_factors(opt) for opt in OPTIONS}
for opt, t in FACT.items():
    print(f"\n[{opt}]"); print(t.assign(원계수=t["원계수"].astype(float).round(2), 보정계수=t["보정계수"].astype(float).round(2)).to_string())


[A 압류 기준]
        원계수    공매n  보정계수
아파트    1.04   73.0  1.19
기타주거   1.50  175.0  1.48
상가·업무  2.02  120.0  1.84
공장·창고  1.50    9.0  1.42
대지     1.59   72.0  1.51
농지·임야  1.95  259.0  1.86
(전체)   1.41  708.0  1.41

[B 기타일반 기준]
        원계수     공매n  보정계수
아파트    1.09   566.0  1.08
기타주거   1.13   758.0  1.12
상가·업무  0.77  1259.0  0.78
공장·창고  1.02    52.0  1.02
대지      NaN     0.0  1.01
농지·임야   NaN     0.0  1.01
(전체)   1.01  2635.0  1.01

[C 전체]
        원계수     공매n  보정계수
아파트    1.09   639.0  1.09
기타주거   1.15   933.0  1.15
상가·업무  0.78  1379.0  0.79
공장·창고  1.03    61.0  1.05
대지     1.59    72.0  1.38
농지·임야  1.95   259.0  1.81
(전체)   1.09  3343.0  1.09


## 4단계. val(9월)로 보정 방식 고르기
후보마다 '9월 공매 실제 낙찰가율 × 계수'가 9월 경매 매각가율과 얼마나 가까운지 봐요. **보정 안 함(계수 1)**도 후보에 넣어서, 보정이 오히려 해로우면 안 하는 쪽을 골라요.
- 건물: A(압류) / B(기타일반) / C(전체) / 보정 안 함
- 토지: 공매 토지는 전부 압류라 A / 보정 안 함

In [ ]:
# 후보에 '보정 안 함(계수 1)'도 넣어서, 보정이 오히려 해가 되면 안 하는 쪽을 고르게 함
# 토지(대지·농지·임야)는 공매 데이터가 전부 압류 → 후보: A(압류 계수) / 보정 안 함
# 건물(아파트·기타주거·상가·업무·공장·창고) → 후보: A / B / C / 보정 안 함(기타일반 기준)
LAND = ["대지", "농지·임야"]
BLDG = ["아파트", "기타주거", "상가·업무", "공장·창고"]
ONES = pd.Series(1.0, index=USAGES + ["(전체)"])
# 후보 이름 → (공매율을 가져올 방식, 계수, 예측 때 고정할 재산유형)
CAND_BLDG = {"A 압류 기준": ("A 압류 기준", FACT["A 압류 기준"]["보정계수"], "압류"),
             "B 기타일반 기준": ("B 기타일반 기준", FACT["B 기타일반 기준"]["보정계수"], "기타일반"),
             "C 전체": ("C 전체", FACT["C 전체"]["보정계수"], "기타일반"),
             "보정 안 함": ("B 기타일반 기준", ONES, "기타일반")}
CAND_LAND = {"A 압류 기준": ("A 압류 기준", FACT["A 압류 기준"]["보정계수"], "압류"),
             "보정 안 함": ("A 압류 기준", ONES, "압류")}

def check(period, cand, auc_region="전국", usages=USAGES):
    rate_opt, fac, _ = cand
    gr, au_ = GONG[(period, rate_opt)], AUC[(period, auc_region)]
    rows = []
    for u in usages:
        if u not in au_.index or pd.isna(au_.loc[u, "경매율"]) or pd.isna(gr.loc[u, "공매율"]) or gr.loc[u, "n"] < 5:
            continue
        pred = gr.loc[u, "공매율"] * float(fac[u])
        rows.append({"용도": u, "공매n": int(gr.loc[u, "n"]), "공매율": gr.loc[u, "공매율"], "계수": float(fac[u]),
                     "예측 경매율": pred, "실제 경매율": au_.loc[u, "경매율"], "경매 매각건수": au_.loc[u, "경매_매각건수"]})
    t = pd.DataFrame(rows)
    if t.empty:
        return t, np.nan
    t["오차(%p)"] = (t["예측 경매율"] - t["실제 경매율"]) * 100
    return t, np.average(np.abs(t["오차(%p)"]), weights=t["경매 매각건수"])

def choose(cands, usages, title):
    # 모든 후보에서 9월 공매가 5건 이상인 용도만 공통으로 비교 (후보마다 다른 용도로 비교하면 불공평)
    common = [u for u in usages if all(GONG[("9월", c[0])].loc[u, "n"] >= 5 for c in cands.values())]
    rows = []
    for name, c in cands.items():
        t, e = check("9월", c, usages=common)
        rows.append({"후보": name, "9월 가중 오차(%p)": e})
        print(f"\n[{title} · {name}] 9월 전국"); print(t.round(3).to_string(index=False))
    sm_ = pd.DataFrame(rows).sort_values("9월 가중 오차(%p)")
    print(f"\n=== val(9월) {title} — 비교 용도 {common} ==="); print(sm_.round(2).to_string(index=False))
    return sm_.iloc[0]["후보"], common

BEST_B, COMMON_B = choose(CAND_BLDG, BLDG, "건물")
BEST_L, COMMON_L = choose(CAND_LAND, LAND, "토지")

FINAL_FACTOR = {u: float(CAND_BLDG[BEST_B][1][u]) for u in BLDG}
FINAL_FACTOR.update({u: float(CAND_LAND[BEST_L][1][u]) for u in LAND})
FINAL_PRPT = {u: CAND_BLDG[BEST_B][2] for u in BLDG}
FINAL_PRPT.update({u: CAND_LAND[BEST_L][2] for u in LAND})
BEST = f"건물 {BEST_B} / 토지 {BEST_L}"
print(f"\n★ 선택: {BEST}")
print(pd.DataFrame({"재산유형 고정": FINAL_PRPT, "보정계수": FINAL_FACTOR}).reindex(USAGES).round(3).to_string())


[건물 · A 압류 기준] 9월 전국
   용도  공매n   공매율    계수  예측 경매율  실제 경매율  경매 매각건수  오차(%p)
  아파트   29 0.711 1.190   0.846   0.842   1240.0   0.357
 기타주거   46 0.632 1.477   0.933   0.600   1946.0  33.364
상가·업무   59 0.353 1.837   0.649   0.496   1498.0  15.384

[건물 · B 기타일반 기준] 9월 전국
   용도  공매n   공매율    계수  예측 경매율  실제 경매율  경매 매각건수  오차(%p)
  아파트   12 0.650 1.085   0.704   0.842   1240.0 -13.772
 기타주거   29 0.650 1.118   0.727   0.600   1946.0  12.732
상가·업무   25 0.609 0.779   0.474   0.496   1498.0  -2.122

[건물 · C 전체] 9월 전국
   용도  공매n   공매율    계수  예측 경매율  실제 경매율  경매 매각건수  오차(%p)
  아파트   41 0.690 1.085   0.749   0.842   1240.0  -9.309
 기타주거   75 0.643 1.149   0.738   0.600   1946.0  13.862
상가·업무   84 0.588 0.787   0.463   0.496   1498.0  -3.295

[건물 · 보정 안 함] 9월 전국
   용도  공매n   공매율  계수  예측 경매율  실제 경매율  경매 매각건수  오차(%p)
  아파트   12 0.650 1.0   0.650   0.842   1240.0 -19.265
 기타주거   29 0.650 1.0   0.650   0.600   1946.0   5.066
상가·업무   25 0.609 1.0   0.609   0.496   1498.0  11.351

=== val(9월) 건물 — 비교 용도 ['

## 5단계. test ① 10월 매각통계
⚠️ 여기부터 test예요. 결과를 보고 4단계 선택을 바꾸지 마세요.

In [ ]:
print("※ 여기부터 test. 결과를 보고 4단계 선택을 바꾸지 마세요.\n")
rows = []
for reg in ["전국", "대구경북"]:
    for name, c in CAND_BLDG.items():
        _, e = check("10월", c, reg, usages=COMMON_B)
        rows.append({"구분": f"건물 · {name}" + (" ★" if name == BEST_B else ""), "경매 지역": reg, "10월 가중 오차(%p)": e})
    for name, c in CAND_LAND.items():
        _, e = check("10월", c, reg, usages=COMMON_L)
        rows.append({"구분": f"토지 · {name}" + (" ★" if name == BEST_L else ""), "경매 지역": reg, "10월 가중 오차(%p)": e})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("※ '10월 공매 실제 낙찰가율 × 계수'로 계수 자체가 맞는지만 봄. 모델 예측까지 포함한 평가는 6~7단계")

※ 여기부터 test. 결과를 보고 4단계 선택을 바꾸지 마세요.

            구분 경매 지역  10월 가중 오차(%p)
  건물 · A 압류 기준    전국          19.60
건물 · B 기타일반 기준    전국          12.68
   건물 · C 전체 ★    전국           4.59
   건물 · 보정 안 함    전국          14.40
  토지 · A 압류 기준    전국          14.83
 토지 · 보정 안 함 ★    전국          13.64
  건물 · A 압류 기준  대구경북          23.57
건물 · B 기타일반 기준  대구경북           3.38
   건물 · C 전체 ★  대구경북           9.20
   건물 · 보정 안 함  대구경북           7.08
  토지 · A 압류 기준  대구경북          17.82
 토지 · 보정 안 함 ★  대구경북           7.20
※ '10월 공매 실제 낙찰가율 × 계수'로 계수 자체가 맞는지만 봄. 모델 예측까지 포함한 평가는 6~7단계


## 6단계. test ② 경매 220건 준비
대구·경북 10월 매각 전수. 아직 공시지가·건축물대장이 없어서 그 피처는 비운 채 **간이 예측**해요. 나중에 `data/auction_features.csv`가 있으면 자동으로 그걸 써요.

In [ ]:
# 220건에는 아직 공시지가·건축물대장이 안 붙어 있음 → 그 피처는 비워 둔 채 예측 (LightGBM은 결측 처리 가능)
# 나중에 V-World로 피처를 붙인 auction_features.csv 가 data 폴더에 있으면 자동으로 그걸 씀
xl = sub(DATA_DIR, "경매_매각사례_시험셋.xlsx")
cases = pd.read_excel(xl, sheet_name="② 사례 기록", header=4)
cases = cases[cases["★ 사건번호"].notna()].copy()
notice = pd.read_excel(xl, sheet_name="① 공고 목록", header=4)
key = ["★ 법원", "★ 사건번호", "★ 물건번호"]
notice = notice.rename(columns={"법원": "★ 법원", "사건번호": "★ 사건번호", "물건번호": "★ 물건번호"})
cases = cases.merge(notice[key + ["법원 표기 용도"]].drop_duplicates(key), on=key, how="left")

def auction_usage(u, court_u):
    if u in ("공장", "창고"): return "공장·창고"
    if u == "상가": return "상가·업무"
    if court_u == "공장용지": return "공장·창고"
    if court_u in ("대지", "잡종지"): return "대지"
    return "농지·임야"                                   # 농지·임야 등

def split_addr(a):
    p = str(a).split()
    sido = p[0] if p else None
    sgg = None
    if len(p) > 1:
        sgg = p[1]
        if len(p) > 2 and p[1].endswith("시") and p[2].endswith("구"):
            sgg = f"{p[1]} {p[2]}"
    return sido, sgg

au = pd.DataFrame({
    "사건": cases["★ 법원"].astype(str) + " " + cases["★ 사건번호"].astype(str) + "-" + cases["★ 물건번호"].astype(int).astype(str),
    "용도": cases["★ 용도"], "usage6": [auction_usage(u, c) for u, c in zip(cases["★ 용도"], cases["법원 표기 용도"])],
    "apslEvlAmt": pd.to_numeric(cases["★ 감정가(원)"], errors="coerce"),
    "매각가": pd.to_numeric(cases["★ 매각가(원)"], errors="coerce"),
    "landSqms": pd.to_numeric(cases["토지면적(㎡)"], errors="coerce"),
    "bldSqms": pd.to_numeric(cases["건물면적(㎡)"], errors="coerce"),
    "opbd_date": pd.to_datetime(cases["★ 매각기일"], errors="coerce"),
    "다필지": cases["★ 소재지 (지번까지)"].astype(str).str.contains("외"),
    "우선표본": cases["메모"].astype(str).str.contains("우선"),
})
au[["sido", "sgg"]] = [split_addr(a) for a in cases["★ 소재지 (지번까지)"]]
au["y"] = au["매각가"] / au["apslEvlAmt"]
au["log_apsl"] = np.log(au["apslEvlAmt"])
au["has_building"] = au["bldSqms"].notna()
au["is_daegu_gb"] = True
au["sido_grp"] = au["sido"]
au["opbd_qtr"] = au["opbd_date"].dt.quarter
for c in ["log_land_price", "apsl_to_official", "bld_age_c", "bld_grnd_flr", "bld_tot_area"]:
    au[c] = np.nan
au["bld_strct"] = np.nan; au["bld_purps"] = np.nan

feat_path = os.path.join(DATA_DIR, "auction_features.csv")
if os.path.exists(feat_path):                         # V-World 피처가 준비되면 덮어씀
    ff = pd.read_csv(feat_path)
    au = au.drop(columns=[c for c in ff.columns if c in au.columns and c != "사건"]).merge(ff, on="사건", how="left")
    print("auction_features.csv 사용 (공시지가·건축물대장 포함)")
else:
    print("auction_features.csv 없음 → 공시지가·건축물대장 없이 간이 예측 (결과는 '간이'로 표기)")

# 지가변동률 (공매 모델과 같은 방식: 매각 월 2개월 전, 그 전 6개월 누적)
LAND_DIR = next((d for d in [sub(PROJECT_DIR, "지역별 지가지수, 지가변동률"), DATA_DIR]
                 if os.path.isdir(d) and "(월) 지역별 지가변동률.csv" in [nfc(x) for x in os.listdir(d)]), None)
def land_file(fname):
    return sub(LAND_DIR, fname)
def read_rone(fname, kind):
    t = pd.read_csv(land_file(fname), encoding="cp949", header=None)
    hm, hk = t.iloc[0], t.iloc[1]; body = t.iloc[3:] if kind == "rate" else t.iloc[2:]
    parts = []
    for j in range(4, t.shape[1]):
        if kind == "rate" and str(hk[j]) != "변동률": continue
        ym = pd.to_datetime(str(hm[j]).replace("년 ", "-").replace("월", ""), format="%Y-%m", errors="coerce")
        if pd.isna(ym): continue
        parts.append(pd.DataFrame({"a": body[1].values, "b": body[2].values, "c": body[3].values,
                                   "ym": ym.to_period("M"), "v": pd.to_numeric(body[j], errors="coerce").values}))
    return pd.concat(parts, ignore_index=True)
SHORT = {"대구광역시": "대구", "경상북도": "경북"}
if LAND_DIR:
    rate = {(r.a, r.b, r.c, r.ym): r.v for r in read_rone("(월) 지역별 지가변동률.csv", "rate").itertuples()}
    idx = {(r.a, r.b, r.c, r.ym): r.v for r in read_rone("(월) 지역별 지가지수.csv", "index").itertuples()}
    lr1, lr6 = [], []
    for s, gname, d in zip(au["sido"], au["sgg"], au["opbd_date"]):
        a = SHORT.get(s, s); m = d.to_period("M") - 2
        p = str(gname).split()
        keys = ([(a, p[0], p[1]), (a, p[0], p[0])] if len(p) > 1 else [(a, p[0], p[0])]) + [(a, a, a)]
        v1 = next((rate[k + (m,)] for k in keys if k + (m,) in rate), np.nan)
        v6 = next(((idx[k + (m,)] / idx[k + (m - 6,)] - 1) * 100 for k in keys if k + (m,) in idx and k + (m - 6,) in idx), np.nan)
        lr1.append(v1); lr6.append(v6)
    au["lr_1m"], au["lr_6m"] = lr1, lr6
else:
    au["lr_1m"] = au["lr_6m"] = np.nan

n_all = len(au)
au = au[au["y"].between(*Y_RANGE)].copy()
print(f"220건 → 낙찰가율 {Y_RANGE[0]}~{Y_RANGE[1]} 밖 {n_all - len(au)}건 제외 → {len(au)}건 평가")
print(au["usage6"].value_counts().to_string())

auction_features.csv 사용 (공시지가·건축물대장 포함)
220건 → 낙찰가율 0.05~1.5 밖 1건 제외 → 219건 평가
usage6
농지·임야    124
상가·업무     49
공장·창고     27
대지        19


## 7단계. 예측 → 보정 → 평가
공매 모델(용도별 재산유형 고정)의 P10/P50/P90에 보정 계수를 곱해서 실제 매각가와 비교해요. 결과는 `data/calib_out/`에 저장돼요.

In [ ]:
meta = json.load(open(os.path.join(MODEL_DIR, "model_meta.json"), encoding="utf-8"))
boosters = {q: lgb.Booster(model_file=os.path.join(MODEL_DIR, f"lgb_q{int(q*100)}.txt")) for q in QS}

def predict_gongmae(frame, prpt_value):
    X = frame[meta["features_num"]].apply(pd.to_numeric, errors="coerce").astype(float)
    f = frame.copy(); f["prpt"] = prpt_value
    for c in meta["features_cat"]:
        X[c] = pd.Categorical(f[c].astype(str).replace({"nan": "없음"}), categories=meta["categories"][c])
    P = np.sort(np.column_stack([boosters[q].predict(X) for q in QS]), axis=1)
    d = frame["usage6"].map(meta.get("cqr_by_usage") or {}).fillna(meta["cqr_delta"]).values
    P[:, 0] -= d; P[:, 2] += d                                          # 공매 모델에서 정한 구간 보정 그대로
    return P

def pinball(y, p, q):
    e = y - p; return np.mean(np.maximum(q * e, (q - 1) * e))

def score(name, y, P):
    P = np.sort(P, axis=1)
    return {"방식": name, "n": len(y), "MAE(P50, %p)": np.mean(np.abs(y - P[:, 1])) * 100,
            "포함률(P10~P90)": np.mean((y >= P[:, 0]) & (y <= P[:, 2])),
            "구간폭(%p)": np.mean(P[:, 2] - P[:, 0]) * 100,
            "pinball": np.mean([pinball(y, P[:, i], q) for i, q in enumerate(QS)]),
            "예측 Σ매각가/Σ감정가": (P[:, 1] * frame_w).sum() / frame_w.sum(), "실제": (y * frame_w).sum() / frame_w.sum()}

y = au["y"].values; frame_w = au["apslEvlAmt"].values
results = []
P0_fix = np.zeros((len(au), 3))
for u in USAGES:                                     # 용도마다 고정할 재산유형으로 공매 모델 예측
    m = (au["usage6"] == u).values
    if m.any():
        P0_fix[m] = predict_gongmae(au[m], FINAL_PRPT[u])
P_final = np.clip(P0_fix * au["usage6"].map(FINAL_FACTOR).values[:, None], 0, None)
results.append(score("보정 없음 (공매 모델 그대로)", y, P0_fix))
results.append(score("★ 보정 (선택한 방식)", y, P_final))
# 참고: 모든 용도를 A(압류)로 했을 때
P0_a = predict_gongmae(au, "압류")
P_a = np.clip(P0_a * au["usage6"].map(FACT["A 압류 기준"]["보정계수"].astype(float)).values[:, None], 0, None)
results.append(score("참고: 전부 A(압류 기준)", y, P_a))
# 기준선: 보정 기간 대구경북 경매 용도별 평균율을 그대로 P50으로 (구간 없음 → 포함률 비교 불가)
dg, nat = AUC[("보정", "대구경북")]["경매율"], AUC[("보정", "전국")]["경매율"]
base = au["usage6"].map(dg).fillna(au["usage6"].map(nat)).values
results.append({"방식": "기준선 (대구경북 경매 평균율)", "n": len(y), "MAE(P50, %p)": np.mean(np.abs(y - base)) * 100,
                "예측 Σ매각가/Σ감정가": (base * frame_w).sum() / frame_w.sum(), "실제": (y * frame_w).sum() / frame_w.sum()})
tag = "" if os.path.exists(feat_path) else " (간이: 공시지가·건축물대장 없음)"
print(f"=== 경매 220건 평가{tag} ===")
print(pd.DataFrame(results).round(3).to_string(index=False))
PRED = {BEST: P_final}

P = np.sort(PRED[BEST], axis=1)
r = au.assign(p10=P[:, 0], p50=P[:, 1], p90=P[:, 2])
def grp(col):
    return r.groupby(col).apply(lambda s: pd.Series({"n": len(s), "MAE(%p)": np.mean(np.abs(s.y - s.p50)) * 100,
                                                     "포함률": np.mean((s.y >= s.p10) & (s.y <= s.p90)),
                                                     "실제 중앙값": s.y.median(), "예측 P50 중앙값": s.p50.median()})).round(3)
print("\n[★ 보정] 용도별"); print(grp("usage6").to_string())
print("\n[★ 보정] 다필지 여부"); print(grp("다필지").to_string())
print("\n[★ 보정] 우선 표본 40건 vs 나머지"); print(grp("우선표본").to_string())

r[["사건", "용도", "usage6", "apslEvlAmt", "매각가", "y", "p10", "p50", "p90", "다필지", "우선표본"]].to_csv(
    os.path.join(OUT_DIR, "auction_predictions.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame({"재산유형 고정": FINAL_PRPT, "보정계수": FINAL_FACTOR}).reindex(USAGES).to_csv(os.path.join(OUT_DIR, "factors_selected.csv"), encoding="utf-8-sig")
json.dump({"building_method": BEST, "fix_prpt_by_usage": FINAL_PRPT,
           "factors": {k: round(v, 4) for k, v in FINAL_FACTOR.items()},
           "shrink_k": SHRINK_K, "note": "경매 예측 = 공매 모델(용도별 재산유형 고정) P10/P50/P90 × 용도별 보정계수"},
          open(os.path.join(OUT_DIR, "calibration.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\n저장: data/calib_out/auction_predictions.csv, factors_selected.csv, calibration.json")

=== 경매 220건 평가 ===
               방식   n  MAE(P50, %p)  포함률(P10~P90)  구간폭(%p)  pinball  예측 Σ매각가/Σ감정가    실제
보정 없음 (공매 모델 그대로) 219        15.416         0.872   61.944    0.050         0.557 0.413
    ★ 보정 (선택한 방식) 219        14.091         0.900   59.713    0.046         0.523 0.413
  참고: 전부 A(압류 기준) 219        26.712         0.831  115.703    0.086         0.572 0.413
기준선 (대구경북 경매 평균율) 219        17.153           NaN      NaN      NaN         0.511 0.413

[★ 보정] 용도별
            n  MAE(%p)    포함률  실제 중앙값  예측 P50 중앙값
usage6                                           
공장·창고    27.0   17.388  0.852   0.531       0.672
농지·임야   124.0   12.228  0.960   0.259       0.287
대지       19.0   16.979  0.895   0.343       0.350
상가·업무    49.0   15.869  0.776   0.355       0.444

[★ 보정] 다필지 여부
           n  MAE(%p)    포함률  실제 중앙값  예측 P50 중앙값
다필지                                             
False  157.0   13.718  0.904   0.349       0.304
True    62.0   15.034  0.887   0.263       0.291

[★ 보정] 우선 표본 40건 